We need to do inference on the test data. Once we have the predictions, we also need to do a posthoc analysis to correctly get the tassel densities for each test image. Another thing to consider here is that we may need to report the mae's seperately for each block, and also entire dataset together.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

2025-06-25 18:27:40.782999: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-25 18:27:40.819624: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-25 18:27:40.819656: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-25 18:27:40.820454: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-25 18:27:40.828346: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load the trained model
basemodel1_stage2 = tf.keras.models.load_model("models/stage2_basemodel1.keras")

2025-06-25 18:27:42.535324: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:06:00.0, compute capability: 7.0


In [3]:
# where is the data?

In [4]:
# The input features are at some other location
test_features_location = "seq_2_seq_test_data"

In [5]:
# os.listdir(test_features_location)

In [6]:
# only get the features files
all_test_featurefile_names = [file for file in os.listdir(test_features_location) if file.split(".")[0][-14:] == 'input_features']
all_test_featurefile_names.sort()

In [7]:
all_test_featurefile_names

['block_0103_extracted_input_features.npy',
 'block_0104_extracted_input_features.npy',
 'block_0105_extracted_input_features.npy',
 'block_0106_extracted_input_features.npy',
 'block_0201_extracted_input_features.npy',
 'block_0202_extracted_input_features.npy',
 'block_0205_extracted_input_features.npy',
 'block_0206_extracted_input_features.npy',
 'block_0302_extracted_input_features.npy',
 'block_0303_extracted_input_features.npy',
 'block_0304_extracted_input_features.npy',
 'block_0305_extracted_input_features.npy',
 'block_0306_extracted_input_features.npy']

In [8]:
# We also need the true tassel counts/densities for the test (and other) data. Let's store these in csv files first?
# For the current data, since our windows are non overlapping, we should be able to get the true counts if we sum across the densities of each image?

density_location = "stacked_densities"

In [9]:
all_density_files = os.listdir(density_location)
all_density_files.sort()

In [10]:
all_density_files

['stacked_densities_block_0101.npy',
 'stacked_densities_block_0102.npy',
 'stacked_densities_block_0103.npy',
 'stacked_densities_block_0104.npy',
 'stacked_densities_block_0105.npy',
 'stacked_densities_block_0106.npy',
 'stacked_densities_block_0201.npy',
 'stacked_densities_block_0202.npy',
 'stacked_densities_block_0203.npy',
 'stacked_densities_block_0204.npy',
 'stacked_densities_block_0205.npy',
 'stacked_densities_block_0206.npy',
 'stacked_densities_block_0301.npy',
 'stacked_densities_block_0302.npy',
 'stacked_densities_block_0303.npy',
 'stacked_densities_block_0304.npy',
 'stacked_densities_block_0305.npy',
 'stacked_densities_block_0306.npy']

In [11]:
# do for one npy density file and later do a function for the rest
example_0 = np.load(os.path.join(density_location, all_density_files[0]))

In [12]:
example_0.shape

(910, 7)

In [13]:
true_densities = np.sum(example_0, axis = 0)

In [14]:
pd.DataFrame(["test_im_" + str(i+1) for i in range(len(true_densities))])

,0
0,test_im_1
1,test_im_2
2,test_im_3
3,test_im_4
4,test_im_5
5,test_im_6
6,test_im_7


In [15]:
# Okay, now save these values in a csv file? 
test_df = pd.concat((pd.DataFrame(["test_im_" + str(i) for i in range(len(true_densities))]), pd.DataFrame(true_densities)), axis = 1)

In [16]:
test_df.columns = ["Test_image_name", "True_density"]

In [17]:
test_df

,Test_image_name,True_density
0,test_im_0,59.989377
1,test_im_1,48.977705
2,test_im_2,55.997329
3,test_im_3,44.000000
4,test_im_4,38.000043
5,test_im_5,46.999915
6,test_im_6,32.005661


In [18]:
# save the dataset
test_df.to_csv(os.path.join("all_true_counts", "true_counts_" + str(all_density_files[0].split(".")[0][-4:]) + ".csv"))

In [19]:
all_density_files[0]

'stacked_densities_block_0101.npy'

In [20]:
# Okay, now define a funtion for this? 

In [21]:
def save_true_densities(density_location, file_name, csv_location):
    # load the file
    loaded_file = np.load(os.path.join(density_location, file_name))
    # count the true densities
    true_densities = np.sum(loaded_file, axis = 0)
    # make a dataframe
    test_df = pd.concat((pd.DataFrame(["test_im_" + str(i) for i in range(len(true_densities))]), pd.DataFrame(true_densities)), axis = 1)
    # give column headers for the dataset
    test_df.columns = ["Test_image_name", "True_density"]
    print(test_df)
    # save the dataset
    test_df.to_csv(os.path.join(csv_location, "true_counts_" + str(file_name.split(".")[0][-4:]) + ".csv"), index = False)
    return test_df

In [22]:
%%time
# try this for all blocks
density_loc = "stacked_densities"
csv_loc = "all_true_counts"
all_blocks_true_dfs = []
for file in all_density_files:
    test_df_returned = save_true_densities(density_loc, file, csv_loc)
    all_blocks_true_dfs.append(test_df_returned)

  Test_image_name  True_density
0       test_im_0     59.989377
1       test_im_1     48.977705
2       test_im_2     55.997329
3       test_im_3     44.000000
4       test_im_4     38.000043
5       test_im_5     46.999915
6       test_im_6     32.005661
  Test_image_name  True_density
0       test_im_0     39.995223
1       test_im_1     48.001213
2       test_im_2     51.000046
3       test_im_3     41.005658
4       test_im_4     38.000174
5       test_im_5     39.003503
6       test_im_6     23.000000
  Test_image_name  True_density
0       test_im_0     40.000661
1       test_im_1     39.000001
2       test_im_2     41.000000
3       test_im_3     31.000000
4       test_im_4     32.000000
5       test_im_5     40.002086
6       test_im_6     27.000176
  Test_image_name  True_density
0       test_im_0     33.000000
1       test_im_1     30.000000
2       test_im_2     39.000001
3       test_im_3     40.000000
4       test_im_4     40.998810
5       test_im_5     42.169009
6       

In [23]:
# We have confirmed the true densities in the csv files

In [24]:
# Okay, what next?

In [25]:
# Predict for test data? And also do a posthoc normalization step (considering the generic case to get the tassel densities per test image)

In [26]:
# get the image height and width
image_height = 768
image_width = 1024
print(image_height, image_width)

768 1024


In [27]:
# all_test_featurefile_names

In [28]:
# Do this for a single block of data?
loaded_test_featured = np.load(os.path.join(test_features_location, all_test_featurefile_names[0]))

In [29]:
loaded_test_featured.shape

(910, 13, 32)

In [30]:
pred_vals_test_im_0 = basemodel1_stage2.predict(loaded_test_featured)

29/29 [==============================] - 1s 4ms/step


2025-06-25 18:27:44.123428: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


In [31]:
pred_vals_test_im_0.shape

(910, 7, 1)

In [32]:
# reshape the predicted value, get rid of the final dimension
pred_vals_test_im_0 = pred_vals_test_im_0.reshape(pred_vals_test_im_0.shape[0], pred_vals_test_im_0.shape[1])

In [33]:
pred_vals_test_im_0.shape

(910, 7)

In [34]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 30, kernel_size = 30):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [35]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'all_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_density']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_density']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_density']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_density']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'all_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, csv_file_name.split(".")[0][-4:] + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

In [36]:
all_csv_files = os.listdir("all_true_counts")
all_csv_files.sort()

In [37]:
all_test_blocks = [file.split(".")[0].split("_")[1] for file in all_test_featurefile_names]
all_test_blocks.sort()

In [38]:
all_test_csv_files = [file for file in all_csv_files if file.split(".")[0].split("_")[-1] in all_test_blocks]
all_test_csv_files.sort()

In [39]:
all_test_csv_files

['true_counts_0103.csv',
 'true_counts_0104.csv',
 'true_counts_0105.csv',
 'true_counts_0106.csv',
 'true_counts_0201.csv',
 'true_counts_0202.csv',
 'true_counts_0205.csv',
 'true_counts_0206.csv',
 'true_counts_0302.csv',
 'true_counts_0303.csv',
 'true_counts_0304.csv',
 'true_counts_0305.csv',
 'true_counts_0306.csv']

In [40]:
preds_block_0103, metrics_0103, preds_df_0103 = get_final_forecasted_and_true_values(pred_vals_test_im_0, image_height, image_width, 30, 30, all_test_csv_files[0])

In [41]:
preds_block_0103

[50.22763431226667,
 48.5403152419108,
 45.61077332272724,
 40.097399247829216,
 33.05866200638142,
 25.429728905873397,
 17.76128263797117]

In [42]:
metrics_0103

[8.335053235980084,
 9.24632730769745,
 PearsonRResult(statistic=0.624833211862317, pvalue=0.13355038228313598),
 -2.1926645324572203]

In [43]:
preds_df_0103

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.000661,50.227634
1,test_im_1,39.000001,48.540315
2,test_im_2,41.000000,45.610773
3,test_im_3,31.000000,40.097399
4,test_im_4,32.000000,33.058662
5,test_im_5,40.002086,25.429729
6,test_im_6,27.000176,17.761283


In [44]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_im_0, axis = 0)

array([50.22763 , 48.540325, 45.61075 , 40.097424, 33.058666, 25.429712,
       17.761292], dtype=float32)

In [45]:
# note the values match

In [46]:
# Now do this for all the test blocks